In [1]:
from app.repositories.vector_repository import connect_to_db
import pandas as pd
import bm25s

c:\Users\T14s\CHATBOT-STAGE\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
conn=connect_to_db()

In [ ]:
conn.rollback()

def fetch_chunks():
    with conn.cursor() as cur:
        cur.execute("""
            SELECT id, chunk_text, lang
            FROM dev.chunks
            ORDER BY id;
        """)
        return cur.fetchall()

rows = fetch_chunks()
ids = [r[0] for r in rows]
texts = [r[1] for r in rows]
langs = [r[2] for r in rows]

In [ ]:
import bm25s
import Stemmer

ar_samples = [t for t, l in zip(texts, langs) if l == "ar"][:3]

stemmer_ar = Stemmer.Stemmer("arabic")

sample_words = ["المواطنين", "الوثيقة", "مواطن"]
print(stemmer_ar.stemWords(sample_words))

tokens = bm25s.tokenize(ar_samples, stemmer=stemmer_ar, stopwords=None)
print(tokens)

In [ ]:
from app.repositories.vector_repository import VectorRepository
repo = VectorRepository()
results=repo.search_lexical(query=""" 
الفصل 83""",lang="ar")

print(results)

In [ ]:
for result in results:
    print(result)

In [ ]:
print(texts[1900])

In [ ]:
repo = VectorRepository(conn=conn)

# Test FR
results_fr = repo.search_lexical(query="le décret de 2023", lang="fr", k=5)
for r in results_fr:
    print(r)

# Test AR 
results_ar = repo.search_lexical(query="ما هي الأنشطة الممنوعة على الموظف العمومي ", lang="ar", k=5)
for r in results_ar:
    print(r)

In [ ]:
from app.core.embeddings import embed_query

query_ar = "ما هي الأنشطة الممنوعة على الموظف العمومي"
emb = embed_query(query_ar)

results_dense_ar = repo.search_similar(embedding=emb, k=5, lang="ar")
for r in results_dense_ar:
        print(r)

In [ ]:
from app.retrieval.hybrid_search import fuse_rrf

fused_ar = fuse_rrf(results_dense_ar, results_ar, k=60) 
for r in fused_ar:
    print(r)


In [ ]:
from app.core.embeddings import embed_query

query_fr = "article 15" 
emb = embed_query(query_fr)

results_dense_fr = repo.search_similar(embedding=emb, k=10, lang="fr")
results_lexical_fr = repo.search_lexical(query=query_fr, lang="fr", k=10)

print("DENSE")
for r in results_dense_fr:
    print(r)

print("LEXICAL")
for r in results_lexical_fr:
    print(r)

print("FUSED (RRF)")
fused = fuse_rrf(results_dense_fr, results_lexical_fr, k=60)
print(len(fused))
for r in fused:
    print(r)

In [ ]:
fused_text=[text[3] for text in fused]

In [ ]:
len(fused_text)

In [4]:
from app.retrieval.reranker import Reranker

rer=Reranker()

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5218.70it/s]


In [ ]:
rer.rerank(query="article 15",candidates=fused_text)

In [7]:
from app.services.retrieval_service import RetrievalService
from app.repositories.vector_repository import VectorRepository

vectrep=VectorRepository()
ret=RetrievalService(vectrep,rer)

ret.search(query="ما هي الأنشطة الممنوعة على الموظف العمومي", lang="ar")

[(1770,
  'قانون الوظيفة العمومية',
  5,
  'لتوظيف المستخدمين الخاضعين لهذا القانون الأساسي العام وترقيتهم \nوتعيينهم ، أو فيما يخص وضعيتهم الإدارية بصفة عامة. \n \nالفصل 15 : مع مراعاة الأحكام التشريعية المنافية الخاصة ببعض\nالهيئات، يمنع على الموظف أن يزاول بصفة مهنية أي نشاط حر أو تابع',
  0.01639344262295082),
 (1901,
  'قانون الوظيفة العمومية',
  30,
  '31 \n \n \nالفصل 83 : ستحدد في مرسوم ميادين النشاط الشخصي التي لا يجوز \nلموظف التعاطي لها نظرا لنوعها ولماهيتها وذلك إذا انقطع نهائيا عن \nعمله أو وقع إيقافه مؤقتا، وتحدد فيه إذا اقتضى الحال آجال منعه من القيام \nبذلك النشاط. وفي حالة المخالفة  لهذا المنع، يمكن أن تقتطع للموظف',
  0.016129032258064516),
 (1775,
  'قانون الوظيفة العمومية',
  6,
  'الفصل16 : يمنع على كل موظف مهما كانت وضعيته،أن تكون له، \nمباشرة أو بواسطة ما أو تحت أي إسم كان، في مقاولة موضوعة تحت \nمراقبة الإدارة أو المصلحة التي ينتمي إليها أو على اتصال بهما، مصالح \nمن شأنها أن تمس بحريته. \n \nالفصل 17 : كل موظف كيفما كانت رتبته في السلك الإداري مسؤول',
  0.01428